如果前面没有 (pytorch_env)，你先手动输入 conda activate pytorch_env

(pytorch_env) C:\Users\YOGA>cd C:\Users\YOGA\PycharmProjects\PythonProject

(pytorch_env) C:\Users\YOGA\PycharmProjects\PythonProject>streamlit run app.py

回车



In [2]:
# -*- coding: utf-8 -*-
"""
质子膜/电池老化电压衰减 GRU 自回归外推 + 面积微积分约束示例
依赖:
    pip install torch pandas numpy scipy matplotlib
"""

import os
import math
import json
import random
from dataclasses import dataclass, asdict
from typing import Optional, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt


# =========================================================
# 1. 配置区：主要改这里
# =========================================================
@dataclass
class CFG:
    # CSV 路径：Windows 桌面示例。请改成你的实际路径。
    csv_path: str = r"C:\Users\YOGA\Desktop\测试集.csv"

    # 列名。若为 None 则自动猜测。
    time_col: Optional[str] = None          # 例如 "time" / "时间" / "h"
    voltage_col: Optional[str] = None       # 例如 "voltage" / "电压" / "V"

    # 如果 CSV 有多个指标，而你想指定某一个作为电压衰减目标，请填列名。
    # 例如 target_col="voltage"
    target_col: Optional[str] = None

    # 数据截断：只使用 <= 920h 的数据训练，然后从 920h 后外推。
    # 如果你的时间单位不是 h，请改成对应单位；若不想截断，设为 None。
    truncate_to_h: Optional[float] = 920.0

    # GRU 训练
    seq_len: int = 50
    epochs: int = 20
    batch_size: int = 128
    hidden_size: int = 64
    lr: float = 1e-3
    seed: int = 42

    # 时间特征放大系数：让网络更重视“时间越长”的特征
    time_feature_scale: float = 20.0

    # 自回归外推步数。10 万步很慢，建议先 2000 步测试。
    max_steps: int = 100_000

    # 面积窗口长度：5000 步
    area_window: int = 5000

    # 基准面积窗口：从 920h 前按“距末端偏移”取窗口。
    # (end_offset, start_offset) 表示窗口 [end-end_offset, end-start_offset)
    # 对应你写的 30000~25000、25000~20000 ... 5000~0
    base_area_offsets: Tuple[Tuple[int, int], ...] = (
        (30000, 25000),
        (25000, 20000),
        (20000, 15000),
        (15000, 10000),
        (10000, 5000),
        (5000, 0),
    )

    # 第一个外推面积窗口相对基准面积的缩小比例
    first_area_shrink_frac: float = 0.01

    # 面积变化率衰减系数：allowed_change <= prev_change * area_change_decay
    area_change_decay: float = 0.98

    # 面积变化量相对当前面积的上限比例，防止一步缩太多
    area_change_cap_frac: float = 0.02

    # 软融合权重：80% 物理面积理论值 + 20% AI 预测值
    w_phys: float = 0.80
    w_ai: float = 0.20

    # 二阶导硬约束模式：
    #   "abs"    : |delta_delta| <= |G(t)| * dt^2，数值稳定，推荐先试
    #   "signed" : delta <= prev_delta + G(t)*dt^2，严格按“不大于G”
    g_mode: str = "abs"
    g_floor: float = 1e-10   # 防止 G 过小导致完全不能变化

    # 输出
    out_csv: str = r"Desktop\gru_extrapolation_result.csv"
    out_fig: str = r"Desktop\gru_extrapolation_result.png"


# =========================================================
# 2. 工具函数
# =========================================================
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def auto_find_col(df: pd.DataFrame, candidates: List[str], exclude: Optional[List[str]] = None):
    exclude = exclude or []
    cols = [c for c in df.columns if c not in exclude]
    for c in cols:
        s = str(c).lower()
        for k in candidates:
            if k.lower() in s:
                return c
    return None


def load_series(cfg: CFG) -> Tuple[np.ndarray, np.ndarray, str, str]:
    df = pd.read_csv(cfg.csv_path)

    # 时间列
    time_col = cfg.time_col
    if time_col is None:
        time_col = auto_find_col(df, ["time", "时间", "hour", "h", "t"])
    if time_col is None:
        df["__time__"] = np.arange(len(df), dtype=float)
        time_col = "__time__"

    # 目标列
    target_col = cfg.target_col or cfg.voltage_col
    if target_col is None:
        target_col = auto_find_col(
            df,
            ["voltage", "volt", "电压", "v", "capacity", "容量"],
            exclude=[time_col],
        )
    if target_col is None:
        # 取第一个非时间数值列
        num_cols = [c for c in df.columns if c != time_col and pd.api.types.is_numeric_dtype(df[c])]
        if not num_cols:
            raise ValueError("CSV 中没有找到可用数值列，请手动指定 target_col/voltage_col")
        target_col = num_cols[0]

    df = df[[time_col, target_col]].copy()
    df.columns = ["time", "value"]
    df["time"] = pd.to_numeric(df["time"], errors="coerce")
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.dropna().sort_values("time")

    # 去重时间点：同一时间取平均
    df = df.groupby("time", as_index=False)["value"].mean()

    # 截断到 920h 前，用于训练和从 920h 开始外推
    if cfg.truncate_to_h is not None:
        if df["time"].max() > cfg.truncate_to_h:
            df = df[df["time"] <= cfg.truncate_to_h].copy()
            print(f"[INFO] 已截断到 <= {cfg.truncate_to_h}，最后时间 = {df['time'].max()}")

    t = df["time"].to_numpy(dtype=np.float64)
    v = df["value"].to_numpy(dtype=np.float64)

    if len(t) < cfg.seq_len + 10:
        raise ValueError(f"数据太短：len={len(t)}，seq_len={cfg.seq_len}")

    print(f"[INFO] 使用列：time={time_col}, value={target_col}")
    print(f"[INFO] 数据长度={len(t)}, 时间范围=[{t.min():.4f}, {t.max():.4f}]")
    return t, v, time_col, target_col


def estimate_dt(t: np.ndarray) -> float:
    d = np.diff(t)
    d = d[np.isfinite(d) & (d > 0)]
    if len(d) == 0:
        return 1.0
    return float(np.median(d))


# =========================================================
# 3. 指数衰减速率拟合 g(t) 与二阶导 G(t)
# =========================================================
def fit_decay_rate_exponential(
    t: np.ndarray,
    v: np.ndarray,
    bin_width: float = 50.0,
):
    """
    先按 50h 切分，得到分箱平均电压。
    衰减速率 r_i = (f(i) - f(i+1)) / 2
    拟合单调非负指数型:
        g(t) = a * exp(-b * (t - t_ref)) + c
        a,b,c >= 0
    g''(t) = a * b^2 * exp(-b*(t-t_ref))
    """
    t0 = t[0]
    bin_id = np.floor((t - t0) / bin_width).astype(int)

    tb, vb = [], []
    for bid in np.unique(bin_id):
        mask = bin_id == bid
        if mask.sum() >= 1:
            tb.append(np.mean(t[mask]))
            vb.append(np.mean(v[mask]))
    tb = np.asarray(tb, dtype=float)
    vb = np.asarray(vb, dtype=float)

    if len(tb) < 4:
        # 回退：常数衰减速率
        r_const = max(np.mean(np.maximum(-np.diff(v), 0.0)), 1e-8)
        t_ref = float(t[0])
        params = np.array([r_const, 1e-8, 1e-10], dtype=float)
        def g_fun(x):
            return np.maximum(params[0] * np.exp(-params[1] * (np.asarray(x) - t_ref)) + params[2], 0.0)
        def G_fun(x):
            return params[0] * params[1] ** 2 * np.exp(-params[1] * (np.asarray(x) - t_ref))
        return g_fun, G_fun, {"params": params.tolist(), "t_ref": t_ref, "rmse": np.nan, "fallback": True}

    # 用户写法：r_i = (f(i) - f(i+1)) / 2
    r = (vb[:-1] - vb[1:]) / 2.0
    tc = (tb[:-1] + tb[1:]) / 2.0
    t_ref = float(tc[0])

    def g_func(x, a, b, c):
        x = np.asarray(x, dtype=float)
        return a * np.exp(-b * (x - t_ref)) + c

    # 初值
    r_max = float(np.nanmax(r)) if np.isfinite(np.nanmax(r)) else 1e-6
    r_min_pos = float(np.nanmax([np.nanmin(r), 0.0]))
    a0 = max(r_max - r_min_pos, 1e-8)
    b0 = 0.01
    c0 = r_min_pos

    bounds = ([0.0, 0.0, 0.0], [np.inf, np.inf, np.inf])

    try:
        popt, _ = curve_fit(
            g_func, tc, r,
            p0=[a0, b0, c0],
            bounds=bounds,
            maxfev=50000,
        )
        pred = g_func(tc, *popt)
        rmse = float(np.sqrt(np.mean((pred - r) ** 2)))
    except Exception as e:
        print(f"[WARN] 指数拟合失败，使用回退常数模型：{e}")
        popt = np.array([max(abs(np.mean(r)), 1e-8), 1e-8, 1e-10], dtype=float)
        pred = g_func(tc, *popt)
        rmse = float(np.sqrt(np.mean((pred - r) ** 2)))

    a, b, c = [float(x) for x in popt]

    def g_fun(x):
        x = np.asarray(x, dtype=float)
        # a,b,c >= 0，因此理论非负；再 max 防止数值误差
        return np.maximum(a * np.exp(-b * (x - t_ref)) + c, 0.0)

    def G_fun(x):
        # g''(t)，非负
        x = np.asarray(x, dtype=float)
        return a * b * b * np.exp(-b * (x - t_ref))

    info = {
        "params_a_b_c": [a, b, c],
        "t_ref": t_ref,
        "rmse": rmse,
        "fallback": False,
        "num_bins": int(len(tb)),
        "num_rate_points": int(len(r)),
    }
    print(f"[INFO] 指数衰减速率拟合完成: a={a:.6e}, b={b:.6e}, c={c:.6e}, RMSE={rmse:.6e}")
    return g_fun, G_fun, info


# =========================================================
# 4. 数据集与 GRU 模型
# =========================================================
class VoltageSeqDataset(Dataset):
    def __init__(
        self,
        t: np.ndarray,
        v: np.ndarray,
        seq_len: int,
        v_mean: float,
        v_std: float,
        delta_std: float,
        t_min: float,
        t_max: float,
        time_scale: float,
    ):
        self.t = t
        self.v = v
        self.seq_len = seq_len
        self.v_mean = v_mean
        self.v_std = max(v_std, 1e-12)
        self.delta_std = max(delta_std, 1e-12)
        self.t_min = t_min
        self.t_span = max(t_max - t_min, 1e-12)
        self.time_scale = time_scale

    def __len__(self):
        return len(self.v) - self.seq_len - 1

    def _vn(self, x):
        return (x - self.v_mean) / self.v_std

    def _tn(self, x):
        return (x - self.t_min) / self.t_span

    def __getitem__(self, idx):
        # 输入 t[idx:idx+seq_len], v[idx:idx+seq_len]
        # 预测 v[idx+seq_len] - v[idx+seq_len-1] 的归一化增量
        v_seq = self.v[idx: idx + self.seq_len]
        t_seq = self.t[idx: idx + self.seq_len]

        x_v = self._vn(v_seq)
        x_t = self._tn(t_seq) * self.time_scale
        x = np.stack([x_v, x_t], axis=1).astype(np.float32)

        delta = self.v[idx + self.seq_len] - self.v[idx + self.seq_len - 1]
        y = np.float32(delta / self.delta_std)
        return torch.from_numpy(x), torch.tensor(y, dtype=torch.float32)


class GRUForecaster(nn.Module):
    def __init__(self, input_size=2, hidden_size=64):
        super().__init__()
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
        )
        self.head = nn.Linear(hidden_size, 1)

    def forward_seq(self, x):
        # x: [B, L, 2]
        out, h = self.gru(x)
        last = out[:, -1, :]
        delta_norm = self.head(last).squeeze(-1)
        return delta_norm, h

    def step(self, x_t, h):
        # x_t: [B, 2], h: [1, B, H]
        out, h = self.gru(x_t.unsqueeze(1), h)
        delta_norm = self.head(out[:, -1, :]).squeeze(-1)
        return delta_norm, h


def train_gru(cfg: CFG, t: np.ndarray, v: np.ndarray, device: torch.device):
    v_mean = float(np.mean(v))
    v_std = float(np.std(v))
    if v_std < 1e-12:
        v_std = 1.0

    dv = np.diff(v)
    delta_std = float(np.std(dv))
    if delta_std < 1e-12:
        delta_std = 1.0

    t_min = float(t.min())
    t_max = float(t.max())

    ds = VoltageSeqDataset(
        t=t, v=v,
        seq_len=cfg.seq_len,
        v_mean=v_mean, v_std=v_std,
        delta_std=delta_std,
        t_min=t_min, t_max=t_max,
        time_scale=cfg.time_feature_scale,
    )
    dl = DataLoader(ds, batch_size=cfg.batch_size, shuffle=True, drop_last=False, num_workers=0)

    model = GRUForecaster(input_size=2, hidden_size=cfg.hidden_size).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    loss_fn = nn.MSELoss()

    model.train()
    for epoch in range(1, cfg.epochs + 1):
        losses = []
        for xb, yb in dl:
            xb = xb.to(device)
            yb = yb.to(device)

            pred, _ = model.forward_seq(xb)
            loss = loss_fn(pred, yb)

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            opt.step()
            losses.append(loss.item())

        print(f"[TRAIN] epoch {epoch:03d}/{cfg.epochs} loss={np.mean(losses):.6e}")

    norm_info = {
        "v_mean": v_mean,
        "v_std": v_std,
        "delta_std": delta_std,
        "t_min": t_min,
        "t_max": t_max,
        "time_scale": cfg.time_feature_scale,
    }
    return model, norm_info


# =========================================================
# 5. 面积基准与面积目标
# =========================================================
def window_area(v: np.ndarray, end_idx: int, end_offset: int, start_offset: int, dt: float) -> Optional[float]:
    """
    计算从 end_idx 往前偏移的窗口面积：
        A = sum(V_t - V_window_start) * dt
    对应窗口 [end_idx-end_offset, end_idx-start_offset)
    """
    start = end_idx - end_offset
    end = end_idx - start_offset
    if start < 0 or end <= start or end > len(v):
        return None
    seg = v[start:end]
    if len(seg) < 2:
        return None
    return float(np.sum(seg - seg[0]) * dt)


def compute_base_area(v: np.ndarray, dt: float, cfg: CFG) -> float:
    end_idx = len(v)
    areas = []
    for end_off, start_off in cfg.base_area_offsets:
        a = window_area(v, end_idx, end_offset=end_off, start_offset=start_off, dt=dt)
        if a is not None and np.isfinite(a):
            areas.append(a)

    if len(areas) == 0:
        # 数据不够长，回退：最后 min(5000, n) 点窗口
        L = min(cfg.area_window, len(v))
        start = len(v) - L
        seg = v[start:]
        areas = [float(np.sum(seg - seg[0]) * dt)]

    base = float(np.mean(areas))
    # 电压下降时面积通常为负；这里保留符号，但目标处会强制负
    print(f"[INFO] 基准面积 base_area={base:.6e}, 使用窗口数={len(areas)}, 各面积={areas}")
    return base


def target_area_for_new_window(
    base_area: float,
    hist_areas: List[float],
    cfg: CFG,
) -> float:
    """
    返回下一个 5000 步窗口的目标面积，强制为负。
    规则：
      1) 第一窗口：|A| 比 |base_area| 小 first_area_shrink_frac；
      2) 后续：|A_k| < |A_{k-1}|；
      3) 面积变化量 | |A_{k-1}| - |A_k| | 不超过上一次变化量；
      4) 变化率逐渐变小：allowed_change <= prev_change * area_change_decay。
    """
    base_abs = max(abs(base_area), 1e-12)

    if len(hist_areas) == 0:
        target_abs = base_abs * (1.0 - cfg.first_area_shrink_frac)
        target_abs = max(target_abs, 0.0)
        return -target_abs

    prev_abs = abs(hist_areas[-1])

    if len(hist_areas) >= 2:
        prev_change = abs(abs(hist_areas[-1]) - abs(hist_areas[-2]))
    else:
        prev_change = base_abs * cfg.first_area_shrink_frac

    # 变化量不能超过上一次变化量，并且变化率逐渐变小
    allowed_change = min(
        prev_change * cfg.area_change_decay,
        prev_abs * cfg.area_change_cap_frac,
        prev_change,
    )
    allowed_change = max(allowed_change, 0.0)

    target_abs = max(prev_abs - allowed_change, 0.0)
    return -target_abs


# =========================================================
# 6. 自回归外推：面积微积分 + 指数 g/G + 80/20 软融合
# =========================================================
@torch.no_grad()
def autoregressive_extrapolation(
    model: GRUForecaster,
    t: np.ndarray,
    v: np.ndarray,
    norm_info: dict,
    g_fun,
    G_fun,
    cfg: CFG,
    device: torch.device,
):
    model.eval()

    dt = estimate_dt(t)
    print(f"[INFO] 估计采样间隔 dt={dt}")

    v_mean = norm_info["v_mean"]
    v_std = norm_info["v_std"]
    delta_std = norm_info["delta_std"]
    t_min = norm_info["t_min"]
    t_max = norm_info["t_max"]
    time_scale = norm_info["time_scale"]
    t_span = max(t_max - t_min, 1e-12)

    def vn(x):
        return (x - v_mean) / v_std

    def tn(x):
        return (x - t_min) / t_span

    # 初始窗口：最后 seq_len 个点
    seq = cfg.seq_len
    v_hist = v[-seq:].copy()
    t_hist = t[-seq:].copy()

    x_seq = np.stack(
        [
            vn(v_hist),
            tn(t_hist) * time_scale,
        ],
        axis=1,
    ).astype(np.float32)
    x_seq_t = torch.from_numpy(x_seq).unsqueeze(0).to(device)  # [1, seq, 2]

    delta_norm, h = model.forward_seq(x_seq_t)
    delta_norm = float(delta_norm.item())

    v_current = float(v[-1])
    t_current = float(t[-1])
    prev_delta = float(v[-1] - v[-2]) if len(v) >= 2 else 0.0

    # 面积基准
    base_area = compute_base_area(v, dt=dt, cfg=cfg)
    hist_areas: List[float] = []

    # 当前面积窗口状态
    window_v0 = v_current
    window_area_accum = 0.0
    window_target_area = 0.0
    window_start_step = 0

    t_ext = []
    v_ext = []
    area_records = []
    target_records = []
    g_records = []

    for step_i in range(cfg.max_steps):
        t_next = t_current + dt

        # 新面积窗口开始
        if step_i % cfg.area_window == 0:
            window_v0 = v_current
            window_area_accum = 0.0
            window_start_step = step_i
            window_target_area = target_area_for_new_window(base_area, hist_areas, cfg)
            print(
                f"[AR] step={step_i:06d}, 新面积窗口, "
                f"target_area={window_target_area:.6e}, "
                f"hist_areas={[round(x, 4) for x in hist_areas[-3:]]}"
            )

        rem_steps = cfg.area_window - (step_i % cfg.area_window)
        rem_steps = max(rem_steps, 1)

        # AI 预测增量：GRU 输出的是归一化增量
        delta_ai = delta_norm * delta_std

        # 物理面积理论增量：
        # 希望本窗口剩余面积按线性分配到剩余步数
        # 目标电压偏移 = (target_area - 已累计面积) / (剩余步数 * dt)
        target_offset = (window_target_area - window_area_accum) / (rem_steps * dt)
        v_phys = window_v0 + target_offset
        delta_phys = v_phys - v_current

        # 80% 物理理论值 + 20% AI 预测值
        delta = cfg.w_phys * delta_phys + cfg.w_ai * delta_ai

        # 二阶导硬约束
        G_val = float(G_fun(np.array([t_next]))[0])
        G_val = max(abs(G_val) if cfg.g_mode == "abs" else G_val, cfg.g_floor)

        if cfg.g_mode == "abs":
            # |V''| <= G  => |delta_delta| <= G * dt^2
            max_ddelta = abs(G_val) * dt * dt
            ddelta = float(np.clip(delta - prev_delta, -max_ddelta, max_ddelta))
            delta = prev_delta + ddelta
        else:
            # 严格 signed：V'' = (delta - prev_delta)/dt^2 <= G
            # => delta <= prev_delta + G * dt^2
            delta = min(delta, prev_delta + G_val * dt * dt)

        # 更新电压
        v_next = v_current + delta

        # 累计本窗口面积：A = sum(V_t - V_window_start)*dt
        window_area_accum += (v_next - window_v0) * dt

        t_ext.append(t_next)
        v_ext.append(v_next)
        area_records.append(window_area_accum)
        target_records.append(window_target_area)
        g_records.append(G_val)

        # 窗口结束：记录实际面积，供下一窗口递减约束
        if (step_i + 1) % cfg.area_window == 0:
            actual_A = float(window_area_accum)
            hist_areas.append(actual_A)
            print(
                f"[AR] step={step_i+1:06d}, 窗口结束, "
                f"actual_area={actual_A:.6e}, target_area={window_target_area:.6e}, "
                f"|actual|={abs(actual_A):.6e}"
            )

        # 为下一步准备 GRU 隐藏状态
        x_next = np.array([[vn(v_next), tn(t_next) * time_scale]], dtype=np.float32)
        x_next_t = torch.from_numpy(x_next).to(device)  # [1, 2]
        delta_norm_t, h = model.step(x_next_t, h)
        delta_norm = float(delta_norm_t.item())

        # 推进状态
        v_current = float(v_next)
        t_current = float(t_next)
        prev_delta = float(delta)

        # 每 1000 步打印一次
        if (step_i + 1) % 1000 == 0:
            print(
                f"[AR] step={step_i+1:06d}/{cfg.max_steps}, "
                f"t={t_current:.4f}, v={v_current:.6f}, "
                f"delta={delta:.6e}, G={G_val:.6e}"
            )

    result = pd.DataFrame(
        {
            "time": t_ext,
            "voltage_pred": v_ext,
            "area_accum_in_window": area_records,
            "target_area": target_records,
            "G_value": g_records,
        }
    )
    return result, base_area, hist_areas


# =========================================================
# 7. 主流程
# =========================================================
def main():
    cfg = CFG()
    set_seed(cfg.seed)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] device={device}")

    # 读取数据
    t, v, time_col, value_col = load_series(cfg)

    # 拟合衰减速率 g 与二阶导 G
    g_fun, G_fun, g_info = fit_decay_rate_exponential(t, v, bin_width=50.0)

    # 训练 GRU
    model, norm_info = train_gru(cfg, t, v, device)

    # 自回归外推
    result, base_area, hist_areas = autoregressive_extrapolation(
        model=model,
        t=t,
        v=v,
        norm_info=norm_info,
        g_fun=g_fun,
        G_fun=G_fun,
        cfg=cfg,
        device=device,
    )

    # 保存结果
    out_csv = cfg.out_csv
    if not os.path.isabs(out_csv):
        out_csv = os.path.join(os.path.expanduser("~"), out_csv)
    os.makedirs(os.path.dirname(out_csv), exist_ok=True)
    result.to_csv(out_csv, index=False, encoding="utf-8-sig")
    print(f"[INFO] 外推结果已保存: {out_csv}")

    # 保存拟合信息
    info_path = out_csv.replace(".csv", "_info.json")
    with open(info_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "cfg": asdict(cfg),
                "g_info": g_info,
                "base_area": base_area,
                "hist_areas": hist_areas,
                "norm_info": norm_info,
            },
            f,
            ensure_ascii=False,
            indent=2,
        )
    print(f"[INFO] 拟合信息已保存: {info_path}")

    # 画图
    fig_path = cfg.out_fig
    if not os.path.isabs(fig_path):
        fig_path = os.path.join(os.path.expanduser("~"), fig_path)
    os.makedirs(os.path.dirname(fig_path), exist_ok=True)

    plt.figure(figsize=(12, 6))
    plt.plot(t, v, label="history / <=920h", linewidth=1.5)
    plt.plot(result["time"], result["voltage_pred"], label="GRU + area/G extrapolation", linewidth=1.2)
    plt.xlabel("time")
    plt.ylabel("voltage")
    plt.title("GRU autoregressive extrapolation with area and second-derivative constraints")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(fig_path, dpi=200)
    plt.close()
    print(f"[INFO] 图已保存: {fig_path}")


if __name__ == "__main__":
    main()

[INFO] device=cpu
[INFO] 使用列：time=Time (h), value=U1 (V)
[INFO] 数据长度=115077, 时间范围=[0.0000, 912.8478]
[INFO] 指数衰减速率拟合完成: a=1.164388e-03, b=1.621548e-03, c=1.430824e-13, RMSE=5.703258e-04
[TRAIN] epoch 001/20 loss=8.238042e-01
[TRAIN] epoch 002/20 loss=6.111604e-01
[TRAIN] epoch 003/20 loss=6.075299e-01
[TRAIN] epoch 004/20 loss=6.070288e-01
[TRAIN] epoch 005/20 loss=6.063613e-01
[TRAIN] epoch 006/20 loss=6.056509e-01
[TRAIN] epoch 007/20 loss=6.053140e-01
[TRAIN] epoch 008/20 loss=6.046685e-01
[TRAIN] epoch 009/20 loss=6.038218e-01
[TRAIN] epoch 010/20 loss=6.033934e-01
[TRAIN] epoch 011/20 loss=6.028649e-01
[TRAIN] epoch 012/20 loss=6.029343e-01
[TRAIN] epoch 013/20 loss=6.025483e-01
[TRAIN] epoch 014/20 loss=6.026124e-01
[TRAIN] epoch 015/20 loss=6.023915e-01
[TRAIN] epoch 016/20 loss=6.018734e-01
[TRAIN] epoch 017/20 loss=6.012809e-01
[TRAIN] epoch 018/20 loss=6.013148e-01
[TRAIN] epoch 019/20 loss=6.010006e-01
[TRAIN] epoch 020/20 loss=6.015033e-01
[INFO] 估计采样间隔 dt=0.008389700000009